# RQ3.2 — Prompt Refinement

This notebook consolidates the four RQ3.2 prompt-refinement stages into a single reproducible analysis:

1. **Stage 1 — Class-definition enrichment**
2. **Stage 2 — Heuristic guidance**
3. **Stage 3 — Reasoning and example-based prompting**
4. **Stage 4 — Confidence-based abstention**

The **project** is the repeated-measures unit and **project-level Macro-F1** is the dependent variable.

Unlike the previous stage-specific notebooks, the experimental conditions are **declared explicitly for each stage**. Extra numeric columns in a CSV are not silently added to the inferential analysis. This is intentional: the statistical design is defined by the experiment, not by whichever numeric columns happen to be present in an input file.

In particular:

- Stage 3 uses five conditions and **does not include `p2_angular_definition`** in its omnibus analysis.
- Stage 4 uses four conditions and **does not include `p2_angular_definition`** in its omnibus analysis.
- Stage 4 coverage/abstention diagnostics are prediction-level outcomes and are not inferred from the Macro-F1 CSVs analyzed here.


Expected replication-package layout for the inputs used here:

```text
results/rq3_2/
├── analysis/
│   └── rq3_2_prompt_refinement.ipynb
└── metrics/
    ├── stage_1_macro_f1.csv
    ├── stage_2_macro_f1.csv
    ├── stage_3_macro_f1.csv
    └── stage_4_macro_f1.csv
```

The loader accepts this layout whether the kernel starts from `analysis/`, `rq3_2/`, `results/`, or the repository root.


In [ ]:
from pathlib import Path
from itertools import combinations
import importlib.util

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import friedmanchisquare

if importlib.util.find_spec("pingouin") is None:
    raise ImportError(
        "This notebook requires pingouin. Install the replication-package "
        "requirements before running it (for example: pip install pingouin)."
    )

import pingouin as pg

ALPHA = 0.05
EXPECTED_PROJECTS = 30
SUBJECT = "project"

# Optional: set this to results/rq3_2 (or another directory containing
# the metrics/ folder) if automatic path resolution does not find the CSVs.
DATA_ROOT_OVERRIDE = None

LABELS = {
    "p0_baseline": "Zero-shot baseline",
    "p1_definition_augmented": "Literature definitions",
    "p2_angular_definition": "Angular definitions",
    "p3_extensive_definition": "Extensive definitions",
    "p4_mix_definition": "Hybrid definitions",
    "p_negative_constraints": "Negative constraints",
    "p_chore_rule_and_negative": "Chore + negative constraints",
    "p_chore_rule": "Chore recovery",
    "p_file_pattern": "File-pattern rules",
    "p_priority_rules": "Priority rules",
    "p_chore_negative_and_dominant": "Chore + negative + dominant intent",
    "p_header_structured": "Structured headers",
    "p_code_disambiguation": "Code disambiguation",
    "p_infra_disambiguation": "Infrastructure disambiguation",
    "p_disambiguation": "Code + infrastructure disambiguation",
    "p_zero-shot_cot": "Zero-shot chain-of-thought",
    "p_few-shot": "Few-shot",
    "p_cot_few-shot": "Few-shot + chain-of-thought",
    "p_exclude_low": "Exclude low confidence",
    "p_high_only": "High confidence only",
}

STAGES = {
    "stage_1_definitions": {
        "title": "Stage 1 — Class-definition enrichment",
        "reference": "p0_baseline",
        "conditions": [
            "p0_baseline",
            "p1_definition_augmented",
            "p2_angular_definition",
            "p3_extensive_definition",
            "p4_mix_definition",
        ],
        "expected_stage_value": "stage_1_definitions",
        "paths": [
            "../metrics/stage_1_macro_f1.csv",
            "metrics/stage_1_macro_f1.csv",
            "rq3_2/metrics/stage_1_macro_f1.csv",
            "results/rq3_2/metrics/stage_1_macro_f1.csv",
        ],
    },
    "stage_2_heuristics": {
        "title": "Stage 2 — Heuristic guidance",
        "reference": "p0_baseline",
        "conditions": [
            "p0_baseline",
            "p2_angular_definition",
            "p_negative_constraints",
            "p_chore_rule_and_negative",
            "p_chore_rule",
            "p_file_pattern",
            "p_priority_rules",
            "p_chore_negative_and_dominant",
            "p_header_structured",
            "p_code_disambiguation",
            "p_infra_disambiguation",
            "p_disambiguation",
        ],
        "expected_stage_value": "stage_2_heuristics",
        "paths": [
            "../metrics/stage_2_macro_f1.csv",
            "metrics/stage_2_macro_f1.csv",
            "rq3_2/metrics/stage_2_macro_f1.csv",
            "results/rq3_2/metrics/stage_2_macro_f1.csv",
        ],
    },
    "stage_3_few_shot_cot": {
        "title": "Stage 3 — Reasoning and example-based prompting",
        "reference": "p_code_disambiguation",
        "conditions": [
            "p0_baseline",
            "p_code_disambiguation",
            "p_zero-shot_cot",
            "p_few-shot",
            "p_cot_few-shot",
        ],
        "expected_stage_value": "stage_3_few_shot_cot",
        "paths": [
            "../metrics/stage_3_macro_f1.csv",
            "metrics/stage_3_macro_f1.csv",
            "rq3_2/metrics/stage_3_macro_f1.csv",
            "results/rq3_2/metrics/stage_3_macro_f1.csv",
        ],
    },
    "stage_4_abstention": {
        "title": "Stage 4 — Confidence-based abstention",
        "reference": "p_code_disambiguation",
        "conditions": [
            "p0_baseline",
            "p_code_disambiguation",
            "p_exclude_low",
            "p_high_only",
        ],
        "expected_stage_value": "stage_4_abstention",
        "paths": [
            "../metrics/stage_4_macro_f1.csv",
            "metrics/stage_4_macro_f1.csv",
            "rq3_2/metrics/stage_4_macro_f1.csv",
            "results/rq3_2/metrics/stage_4_macro_f1.csv",
        ],
    },
}

def _search_roots():
    roots = []
    if DATA_ROOT_OVERRIDE is not None:
        roots.append(Path(DATA_ROOT_OVERRIDE).expanduser().resolve())

    cwd = Path.cwd().resolve()
    roots.append(cwd)
    roots.extend(list(cwd.parents)[:5])

    unique = []
    seen = set()
    for root in roots:
        marker = str(root)
        if marker not in seen:
            unique.append(root)
            seen.add(marker)
    return unique


def resolve_stage_file(stage_key):
    cfg = STAGES[stage_key]
    attempted = []

    for root in _search_roots():
        for relative in cfg["paths"]:
            candidate = (root / relative).resolve()
            attempted.append(candidate)
            if candidate.is_file():
                return candidate

    formatted = "\n".join(f"  - {path}" for path in attempted)
    raise FileNotFoundError(
        f"Could not locate the Macro-F1 CSV for {stage_key}.\n"
        f"Paths checked:\n{formatted}\n\n"
        "Set DATA_ROOT_OVERRIDE explicitly if your replication package uses "
        "a different directory layout."
    )


def load_stage_data(stage_key):
    cfg = STAGES[stage_key]
    path = resolve_stage_file(stage_key)
    df = pd.read_csv(path)

    required = [SUBJECT, *cfg["conditions"]]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(
            f"{stage_key}: missing required columns {missing} in {path}"
        )

    if df[SUBJECT].duplicated().any():
        duplicates = (
            df.loc[df[SUBJECT].duplicated(), SUBJECT]
            .astype(str)
            .tolist()
        )
        raise ValueError(
            f"{stage_key}: each project must occur once. "
            f"Duplicates: {duplicates}"
        )

    if len(df) != EXPECTED_PROJECTS or df[SUBJECT].nunique() != EXPECTED_PROJECTS:
        raise ValueError(
            f"{stage_key}: expected {EXPECTED_PROJECTS} projects, "
            f"found {len(df)} rows and {df[SUBJECT].nunique()} unique projects."
        )

    for condition in cfg["conditions"]:
        converted = pd.to_numeric(df[condition], errors="coerce")
        if converted.isna().any():
            bad_projects = df.loc[converted.isna(), SUBJECT].astype(str).tolist()
            raise ValueError(
                f"{stage_key}: non-numeric or missing values in {condition}; "
                f"projects: {bad_projects}"
            )
        df[condition] = converted

    if "stage" in df.columns:
        observed = set(df["stage"].dropna().astype(str))
        expected = cfg["expected_stage_value"]
        if observed and observed != {expected}:
            raise ValueError(
                f"{stage_key}: expected stage column value {expected!r}, "
                f"observed {sorted(observed)}"
            )

    analyzed = set(cfg["conditions"])
    ignored_numeric = []
    for column in df.columns:
        if column in analyzed or column in {SUBJECT, "stage"}:
            continue
        converted = pd.to_numeric(df[column], errors="coerce")
        original_non_missing = df[column].notna()
        if (
            original_non_missing.any()
            and converted.loc[original_non_missing].notna().all()
        ):
            ignored_numeric.append(column)

    print(f"Data file: {path}")
    print(f"Projects: {df[SUBJECT].nunique()}")
    print(f"Analyzed conditions ({len(cfg['conditions'])}): {cfg['conditions']}")
    if ignored_numeric:
        print(
            "Extra numeric columns intentionally excluded from this stage: "
            f"{ignored_numeric}"
        )

    columns = ["stage", SUBJECT, *cfg["conditions"]]
    return df[[column for column in columns if column in df.columns]].copy()


## Statistical workflow

The same prespecified repeated-measures workflow is applied to every stage:

1. report descriptive statistics for the explicitly declared configurations;
2. apply Shapiro–Wilk to each configuration;
3. if normality is not rejected for all configurations, use the parametric route:
   - Mauchly's test of sphericity;
   - repeated-measures ANOVA;
   - Greenhouse–Geisser correction when sphericity is violated;
   - Holm-corrected paired Student's \(t\)-tests when the omnibus test is significant;
   - Hedges' \(g_{av}\) as the paired standardized effect size;
4. otherwise, use the non-parametric route:
   - Friedman test;
   - Holm-corrected Wilcoxon signed-rank tests when the omnibus test is significant;
   - matched-pairs rank-biserial correlation (\(r_{rb}\));
5. pairwise post-hoc tests are executed only after a significant omnibus test.

All decisions use unrounded values. Rounding is applied only for display.


In [ ]:
def _format_decimal(value, digits=3):
    value = float(value)

    if value != 0 and abs(value) < 10 ** (-digits):
        return f"<.{''.join(['0'] * (digits - 1))}1"

    text = f"{value:.{digits}f}"
    if text.startswith("0."):
        return text[1:]
    if text.startswith("-0."):
        return "-" + text[2:]
    return text


def _format_p(p_value):
    p_value = float(p_value)
    if p_value < 0.001:
        return r"$p<.001$"
    return f"$p={_format_decimal(p_value)}$"


def _pretty_label(configuration):
    if configuration in LABELS:
        return LABELS[configuration]
    return configuration.replace("_", " ").replace("-", " ").strip().title()


def _run_parametric_route(
    long_data,
    analysis_data,
    subject,
    conditions,
    dv_name,
    alpha,
):
    n_subjects = analysis_data[subject].nunique()
    k = len(conditions)

    sphericity = pg.sphericity(
        data=long_data,
        dv=dv_name,
        within="configuration",
        subject=subject,
    )
    apply_gg = bool(sphericity.pval < alpha)

    anova = pg.rm_anova(
        data=long_data,
        dv=dv_name,
        within="configuration",
        subject=subject,
        correction=True,
        detailed=True,
    )
    effect = anova.iloc[0]

    epsilon = float(effect["eps"])
    df1_uncorrected = k - 1
    df2_uncorrected = (k - 1) * (n_subjects - 1)

    if apply_gg:
        final_p = float(effect["p_GG_corr"])
        final_df1 = epsilon * df1_uncorrected
        final_df2 = epsilon * df2_uncorrected
        correction = "Greenhouse-Geisser"
    else:
        final_p = float(effect["p_unc"])
        final_df1 = float(df1_uncorrected)
        final_df2 = float(df2_uncorrected)
        correction = "None"

    omnibus = {
        "test": "Repeated-measures ANOVA",
        "statistic_name": "F",
        "statistic": float(effect["F"]),
        "df1": final_df1,
        "df2": final_df2,
        "p": final_p,
        "effect_name": "eta_G_squared",
        "effect": float(effect["ng2"]),
        "epsilon": epsilon,
        "correction": correction,
        "significant": final_p < alpha,
    }

    posthoc = None
    if final_p < alpha:
        posthoc = pg.pairwise_tests(
            data=long_data,
            dv=dv_name,
            within="configuration",
            subject=subject,
            parametric=True,
            padjust="holm",
            effsize="hedges",
            return_desc=True,
        )
        posthoc = posthoc[[
            "A", "B", "mean_A", "mean_B", "T", "dof",
            "p_unc", "p_corr", "p_adjust", "hedges",
        ]].copy()
        posthoc["significant"] = posthoc["p_corr"] < alpha

    return {
        "sphericity": sphericity,
        "anova": anova,
        "omnibus": omnibus,
        "posthoc": posthoc,
    }


def _run_nonparametric_route(
    analysis_data,
    conditions,
    alpha,
):
    n_subjects = len(analysis_data)
    k = len(conditions)

    friedman = friedmanchisquare(
        *[analysis_data[condition].to_numpy() for condition in conditions]
    )
    q_statistic = float(friedman.statistic)
    p_value = float(friedman.pvalue)
    kendalls_w = q_statistic / (n_subjects * (k - 1))

    omnibus = {
        "test": "Friedman",
        "statistic_name": "Q",
        "statistic": q_statistic,
        "df1": float(k - 1),
        "df2": None,
        "p": p_value,
        "effect_name": "Kendalls_W",
        "effect": kendalls_w,
        "epsilon": None,
        "correction": "Not applicable",
        "significant": p_value < alpha,
    }

    posthoc = None
    if p_value < alpha:
        rows = []

        for configuration_a, configuration_b in combinations(conditions, 2):
            values_a = analysis_data[configuration_a].to_numpy()
            values_b = analysis_data[configuration_b].to_numpy()

            if np.allclose(values_a - values_b, 0):
                w_value = 0.0
                pair_p = 1.0
                rbc = 0.0
            else:
                wilcoxon = pg.wilcoxon(
                    values_a,
                    values_b,
                    alternative="two-sided",
                ).iloc[0]
                w_value = float(wilcoxon["W-val"])
                pair_p = float(wilcoxon["p-val"])
                rbc = float(wilcoxon["RBC"])

            rows.append({
                "A": configuration_a,
                "B": configuration_b,
                "mean_A": float(np.mean(values_a)),
                "mean_B": float(np.mean(values_b)),
                "W": w_value,
                "p_unc": pair_p,
                "RBC": rbc,
            })

        posthoc = pd.DataFrame(rows)
        reject, corrected_p = pg.multicomp(
            posthoc["p_unc"].to_numpy(),
            alpha=alpha,
            method="holm",
        )
        posthoc["p_corr"] = corrected_p
        posthoc["significant"] = reject

    return {
        "sphericity": None,
        "anova": None,
        "omnibus": omnibus,
        "posthoc": posthoc,
    }


def run_repeated_measures_analysis(
    data,
    subject,
    conditions,
    dv_name="macro_f1",
    alpha=ALPHA,
):
    if len(conditions) < 3:
        raise ValueError("At least three configurations are required.")

    analysis_data = data[[subject, *conditions]].copy()

    if analysis_data.isna().any().any():
        raise ValueError(
            "The stage contains missing values. The RQ3.2 analysis expects "
            "the same complete set of 30 projects for every configuration."
        )

    long_data = analysis_data.melt(
        id_vars=subject,
        value_vars=conditions,
        var_name="configuration",
        value_name=dv_name,
    )

    descriptive = (
        long_data.groupby("configuration")[dv_name]
        .agg(N="count", Mean="mean", SD="std")
        .sort_values("Mean", ascending=False)
    )

    shapiro_rows = []
    for condition in conditions:
        normality = pg.normality(
            analysis_data[condition],
            method="shapiro",
            alpha=alpha,
        ).iloc[0]

        shapiro_rows.append({
            "Configuration": condition,
            "N": int(analysis_data[condition].notna().sum()),
            "W": float(normality["W"]),
            "p": float(normality["pval"]),
            "Normality not rejected": bool(normality["normal"]),
        })

    shapiro = pd.DataFrame(shapiro_rows).set_index("Configuration")
    all_normal = bool(shapiro["Normality not rejected"].all())

    if all_normal:
        route = "parametric"
        inferential = _run_parametric_route(
            long_data=long_data,
            analysis_data=analysis_data,
            subject=subject,
            conditions=conditions,
            dv_name=dv_name,
            alpha=alpha,
        )
    else:
        route = "non-parametric"
        inferential = _run_nonparametric_route(
            analysis_data=analysis_data,
            conditions=conditions,
            alpha=alpha,
        )

    posthoc = inferential["posthoc"]
    significant_posthoc = None
    if posthoc is not None:
        significant_posthoc = posthoc.loc[posthoc["significant"]].copy()

    print("DESCRIPTIVE STATISTICS")
    display(descriptive.round({"Mean": 4, "SD": 4}))

    print("SHAPIRO-WILK TESTS")
    display(shapiro.round({"W": 4, "p": 4}))

    print(f"SELECTED ROUTE: {route}")

    if inferential["sphericity"] is not None:
        sphericity = inferential["sphericity"]
        display(pd.DataFrame([{
            "Mauchly W": sphericity.W,
            "chi-square": sphericity.chi2,
            "df": sphericity.dof,
            "p": sphericity.pval,
            "Sphericity": bool(sphericity.spher),
        }]).round(4))

    display(pd.DataFrame([inferential["omnibus"]]).round(4))

    if posthoc is None:
        print(
            "The omnibus test was not significant; "
            "post-hoc comparisons were not performed."
        )
    elif significant_posthoc.empty:
        print(
            "No pairwise comparison remained significant after Holm correction."
        )
    else:
        print("SIGNIFICANT HOLM-CORRECTED COMPARISONS")
        display(significant_posthoc.round(4))

    return {
        "route": route,
        "analysis_data": analysis_data,
        "long_data": long_data,
        "descriptive": descriptive,
        "shapiro": shapiro,
        "sphericity": inferential["sphericity"],
        "anova": inferential["anova"],
        "omnibus": inferential["omnibus"],
        "posthoc": posthoc,
        "significant_posthoc": significant_posthoc,
        "alpha": alpha,
    }


def generate_dynamic_report(
    results,
    report_title,
    reference=None,
):
    descriptive = results["descriptive"]
    shapiro = results["shapiro"]
    omnibus = results["omnibus"]
    posthoc = results["posthoc"]
    significant = results["significant_posthoc"]
    route = results["route"]

    ranking_lines = []
    for configuration, row in descriptive.iterrows():
        ranking_lines.append(
            f"- **{_pretty_label(configuration)}**: "
            f"mean Macro-F1 = {row['Mean']:.3f}, SD = {row['SD']:.3f}"
        )

    non_normal = shapiro.index[
        ~shapiro["Normality not rejected"]
    ].tolist()

    if route == "parametric":
        normality_text = (
            "Shapiro–Wilk did not reject normality for any configuration; "
            "therefore, the parametric repeated-measures route was selected."
        )

        sphericity = results["sphericity"]
        if sphericity.pval < results["alpha"]:
            sphericity_text = (
                "Mauchly's test indicated a violation of sphericity "
                f"($W={_format_decimal(sphericity.W)}$, "
                f"{_format_p(sphericity.pval)}); the Greenhouse–Geisser "
                "correction was used."
            )
        else:
            sphericity_text = (
                "Mauchly's test did not indicate a violation of sphericity "
                f"($W={_format_decimal(sphericity.W)}$, "
                f"{_format_p(sphericity.pval)})."
            )

        omnibus_text = (
            "The repeated-measures ANOVA "
            + ("detected" if omnibus["significant"] else "did not detect")
            + " an overall configuration effect, "
            + f"$F({omnibus['df1']:.2f},{omnibus['df2']:.2f})="
            + f"{omnibus['statistic']:.2f}$, {_format_p(omnibus['p'])}, "
            + f"$\\eta_G^2={_format_decimal(omnibus['effect'])}$."
        )

        effect_column = "hedges"
        effect_label = "Hedges' $g_{av}$"
        statistic_column = "T"
        statistic_label = "$t$"

    else:
        normal_names = ", ".join(
            _pretty_label(configuration)
            for configuration in non_normal
        )
        normality_text = (
            "Shapiro–Wilk rejected normality for at least one configuration "
            f"({normal_names}); therefore, the non-parametric route was selected."
        )
        sphericity_text = (
            "Mauchly's test is not required for the Friedman route."
        )
        omnibus_text = (
            "The Friedman test "
            + ("detected" if omnibus["significant"] else "did not detect")
            + " an overall configuration effect, "
            + f"$\\chi^2({int(omnibus['df1'])})={omnibus['statistic']:.2f}$, "
            + f"{_format_p(omnibus['p'])}, "
            + f"Kendall's $W={_format_decimal(omnibus['effect'])}$."
        )

        effect_column = "RBC"
        effect_label = "$r_{rb}$"
        statistic_column = "W"
        statistic_label = "$W$"

    if posthoc is None:
        pairwise_text = (
            "Because the omnibus test was not significant, confirmatory "
            "pairwise post-hoc tests were not performed."
        )

    elif significant is None or significant.empty:
        pairwise_text = (
            "No pairwise comparison remained significant after Holm correction."
        )

    else:
        pairwise_lines = []

        for _, row in significant.iterrows():
            if row["mean_A"] >= row["mean_B"]:
                better = row["A"]
                worse = row["B"]
                oriented_statistic = row[statistic_column]
            else:
                better = row["B"]
                worse = row["A"]
                oriented_statistic = (
                    -row[statistic_column]
                    if statistic_column == "T"
                    else row[statistic_column]
                )

            pairwise_lines.append(
                f"- **{_pretty_label(better)}** outperformed "
                f"**{_pretty_label(worse)}** "
                f"({statistic_label} = {oriented_statistic:.2f}, "
                f"{_format_p(row['p_corr'])}, "
                f"{effect_label} = {abs(row[effect_column]):.3f})"
            )

        pairwise_text = (
            "The following comparisons remained significant after "
            "Holm correction:\n\n" + "\n".join(pairwise_lines)
        )

    reference_text = ""

    if reference and reference in descriptive.index:
        reference_label = _pretty_label(reference)

        if posthoc is None:
            reference_text = (
                f"No confirmatory pairwise claim relative to "
                f"**{reference_label}** is made because the omnibus test "
                "was not significant."
            )
        else:
            improved = []
            worsened = []

            for _, row in significant.iterrows():
                if reference not in {row["A"], row["B"]}:
                    continue

                other = row["B"] if row["A"] == reference else row["A"]

                if (
                    descriptive.loc[other, "Mean"]
                    > descriptive.loc[reference, "Mean"]
                ):
                    improved.append(_pretty_label(other))
                else:
                    worsened.append(_pretty_label(other))

            parts = []

            if improved:
                parts.append(
                    "Significantly higher than the reference: "
                    + ", ".join(improved)
                    + "."
                )

            if worsened:
                parts.append(
                    "Significantly lower than the reference: "
                    + ", ".join(worsened)
                    + "."
                )

            if not parts:
                parts.append(
                    f"No configuration differed significantly from "
                    f"**{reference_label}** after Holm correction."
                )

            reference_text = " ".join(parts)

    best_configuration = descriptive.index[0]
    best_label = _pretty_label(best_configuration)

    if omnibus["significant"]:
        conclusion = (
            f"**{best_label}** had the highest descriptive mean. The omnibus "
            "test detected differences among configurations; statistical "
            "claims about specific pairs are limited to the corrected "
            "comparisons reported above."
        )
    else:
        conclusion = (
            f"**{best_label}** had the highest descriptive mean, but the "
            "omnibus test did not detect an overall difference. The observed "
            "ordering should therefore be interpreted descriptively."
        )

    return f"""
## {report_title}

### Descriptive results

{chr(10).join(ranking_lines)}

### Assumptions and omnibus test

{normality_text} {sphericity_text} {omnibus_text}

### Pairwise evidence

{pairwise_text}

{reference_text}

### Data-driven conclusion

{conclusion}
"""


def analyze_stage(stage_key):
    cfg = STAGES[stage_key]
    df = load_stage_data(stage_key)

    results = run_repeated_measures_analysis(
        data=df,
        subject=SUBJECT,
        conditions=cfg["conditions"],
    )

    report = generate_dynamic_report(
        results=results,
        report_title=f"{cfg['title']} — interpretation generated from current data",
        reference=cfg["reference"],
    )

    display(Markdown(report))
    return df, results


## Stage 1 — Class-definition enrichment

In [ ]:
stage_1_definitions_data, stage_1_definitions_results = analyze_stage("stage_1_definitions")

## Stage 2 — Heuristic guidance

In [ ]:
stage_2_heuristics_data, stage_2_heuristics_results = analyze_stage("stage_2_heuristics")

## Stage 3 — Reasoning and example-based prompting

In [ ]:
stage_3_few_shot_cot_data, stage_3_few_shot_cot_results = analyze_stage("stage_3_few_shot_cot")

## Stage 4 — Confidence-based abstention

In [ ]:
stage_4_abstention_data, stage_4_abstention_results = analyze_stage("stage_4_abstention")

## Cross-stage checkpoint summary

This table reproduces the principal descriptive checkpoints used to summarize RQ3.2 in the paper. Differences from the baseline are descriptive only.

The selected checkpoint rows are not an additional inferential analysis; they summarize means and standard deviations already computed within the corresponding stages.


In [ ]:
STAGE_RESULTS = {
    "stage_1_definitions": stage_1_definitions_results,
    "stage_2_heuristics": stage_2_heuristics_results,
    "stage_3_few_shot_cot": stage_3_few_shot_cot_results,
    "stage_4_abstention": stage_4_abstention_results,
}

baseline_means = [
    results["descriptive"].loc["p0_baseline", "Mean"]
    for results in STAGE_RESULTS.values()
]

if not np.allclose(baseline_means, baseline_means[0], atol=1e-12, rtol=0):
    raise RuntimeError(
        f"Baseline means differ across stage CSVs: {baseline_means}"
    )

baseline_mean = float(baseline_means[0])

CHECKPOINTS = [
    ("Baseline", "stage_1_definitions", "p0_baseline"),
    ("Definitions", "stage_1_definitions", "p2_angular_definition"),
    ("Heuristics", "stage_2_heuristics", "p_code_disambiguation"),
    ("Reasoning", "stage_3_few_shot_cot", "p_zero-shot_cot"),
    ("Examples", "stage_3_few_shot_cot", "p_few-shot"),
    ("Combined", "stage_3_few_shot_cot", "p_cot_few-shot"),
    ("Abstention", "stage_4_abstention", "p_exclude_low"),
    ("Abstention", "stage_4_abstention", "p_high_only"),
]

rows = []

for checkpoint, stage_key, configuration in CHECKPOINTS:
    descriptive = STAGE_RESULTS[stage_key]["descriptive"]
    row = descriptive.loc[configuration]

    rows.append({
        "Checkpoint": checkpoint,
        "Configuration": _pretty_label(configuration),
        "Mean Macro-F1": float(row["Mean"]),
        "SD": float(row["SD"]),
        "Delta vs. baseline": (
            np.nan
            if configuration == "p0_baseline"
            else float(row["Mean"] - baseline_mean)
        ),
    })

checkpoint_summary = pd.DataFrame(rows)

formatted = checkpoint_summary.copy()
formatted["Mean Macro-F1"] = formatted["Mean Macro-F1"].map(lambda x: f"{x:.3f}")
formatted["SD"] = formatted["SD"].map(lambda x: f"{x:.3f}")
formatted["Delta vs. baseline"] = formatted["Delta vs. baseline"].map(
    lambda x: "---" if pd.isna(x) else f"{x:+.3f}"
)

display(formatted)


## Reproducibility notes

- The notebook consumes **local CSV files**; it does not download data from Google Drive and does not install packages dynamically.
- Experimental conditions are declared explicitly in `STAGES`.
- Missing required conditions, duplicate projects, non-numeric values, or an unexpected number of projects cause the notebook to fail rather than silently changing the analysis.
- Extra numeric columns are reported and ignored unless they are explicitly part of that stage's design.
- Stage 3 intentionally analyzes five configurations: baseline, code disambiguation, zero-shot CoT, few-shot, and few-shot + CoT.
- Stage 4 intentionally analyzes four configurations: baseline, code disambiguation, exclude-low, and high-only.
- Coverage, per-seed filtering, final abstention counts, and confidence-stability diagnostics require the prediction-level RQ3.2 outputs and are outside the scope of these project-level Macro-F1 CSVs.
- The generated narrative is a consistency aid. Scientific interpretation should still be reviewed before copying text into the paper.
